# Individual-stock Granger causality tests

This notebook independently tests whether the hourly `AI_Risk_total` index predicts each of the 20 stocks. It is separate from both the pooled-panel model and the equal-weight portfolio model.

In [1]:
from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6g}")

ROOT = Path.cwd().resolve()
if not (ROOT / "download_hourly_returns.py").exists():
    raise RuntimeError("Run this notebook from the project root directory.")

REGULAR_PATH = ROOT / "data" / "hourly_10_16" / "hourly_returns_long.csv"
BRIDGE_PATH = ROOT / "data" / "premarket_bridge_07_10" / "hourly_returns_long.csv"
RISK_PATH = ROOT / "data" / "hourly_ai_risk_index" / "x" / "hourly_ai_risk_index.csv"

for path in (REGULAR_PATH, BRIDGE_PATH, RISK_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

environment = pd.DataFrame(
    {
        "component": ["Python", "pandas", "NumPy", "SciPy", "statsmodels", "platform"],
        "version": [
            platform.python_version(),
            pd.__version__,
            np.__version__,
            scipy.__version__,
            statsmodels.__version__,
            platform.platform(),
        ],
    }
)
display(environment)
print("Interpreter:", sys.executable)
print("Project root:", ROOT)

,component,version
0,Python,3.14.7
1,pandas,3.0.6
2,NumPy,2.5.3
3,SciPy,1.18.1
4,statsmodels,0.15.0
5,platform,Windows-11-10.0.26200-SP0


Interpreter: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development\.venv\Scripts\python.exe
Project root: C:\Users\Jerry Cai\Desktop\NLP_Assignment4_AI_risk_and_AI_development


## 1. Load, validate and align returns with the risk index

In [2]:
def parse_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().eq("true")


def load_return_file(path: Path, expected_mode: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {
        "ticker", "trading_date", "bar_start_utc", "bar_end_utc",
        "duration_minutes", "window_complete", "ohlc_valid", "nonpositive_price",
        "split_event", "log_ret_oc", "return_requires_review",
    }
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    frame["bar_start_utc"] = pd.to_datetime(frame["bar_start_utc"], utc=True)
    frame["bar_end_utc"] = pd.to_datetime(frame["bar_end_utc"], utc=True)
    frame["window_complete"] = parse_bool(frame["window_complete"])
    frame["ohlc_valid"] = parse_bool(frame["ohlc_valid"])
    frame["nonpositive_price"] = parse_bool(frame["nonpositive_price"])
    frame["split_event"] = parse_bool(frame["split_event"])
    frame["return_requires_review"] = parse_bool(frame["return_requires_review"])
    frame["duration_minutes"] = pd.to_numeric(frame["duration_minutes"], errors="coerce")
    frame["log_ret_oc"] = pd.to_numeric(frame["log_ret_oc"], errors="coerce")
    if "window_mode" in frame.columns and set(frame["window_mode"].dropna().unique()) != {expected_mode}:
        raise ValueError(f"Unexpected window mode in {path}")
    if frame.duplicated(["ticker", "bar_start_utc"]).any():
        raise ValueError(f"Duplicate ticker/start timestamps in {path}")
    return frame


regular = load_return_file(REGULAR_PATH, "clock_10_16")
bridge = load_return_file(BRIDGE_PATH, "bridge_07_10")
risk = pd.read_csv(RISK_PATH)

risk_required = {
    "hour", "AI_Risk_total", "coverage_status", "coverage_complete",
    "is_backtest_eligible_observed_sample", "is_backtest_eligible_strict",
}
missing = risk_required.difference(risk.columns)
if missing:
    raise ValueError(f"Risk index is missing columns: {sorted(missing)}")

risk["hour"] = pd.to_datetime(risk["hour"], utc=True)
risk["AI_Risk_total"] = pd.to_numeric(risk["AI_Risk_total"], errors="coerce")
risk["observed_eligible"] = parse_bool(risk["is_backtest_eligible_observed_sample"])
risk["strict_eligible"] = parse_bool(risk["is_backtest_eligible_strict"])
if risk["hour"].duplicated().any():
    raise ValueError("Risk index contains duplicate hourly timestamps")
if len(risk) > 1 and not risk["hour"].sort_values().diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("Risk index is not a continuous hourly grid")

regular_hours = set(regular["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
bridge_hours = set(bridge["bar_start_utc"].dt.tz_convert("America/New_York").dt.hour.unique())
assert regular_hours == set(range(10, 16)), regular_hours
assert bridge_hours == {7, 8, 9}, bridge_hours
assert regular["duration_minutes"].eq(60).all()

targets = regular.loc[
    regular["window_complete"]
    & regular["ohlc_valid"]
    & ~regular["nonpositive_price"]
    & ~regular["split_event"]
    & ~regular["return_requires_review"]
    & regular["log_ret_oc"].notna()
].copy()
targets = targets.rename(columns={"log_ret_oc": "target_return"})
targets["target_start_ny"] = targets["bar_start_utc"].dt.tz_convert("America/New_York")
targets["target_end_ny"] = targets["bar_end_utc"].dt.tz_convert("America/New_York")
targets["target_hour"] = targets["target_start_ny"].dt.hour.astype(str).str.zfill(2) + ":00"
targets["trading_date"] = targets["trading_date"].astype(str)
targets["weekday"] = pd.to_datetime(targets["trading_date"]).dt.day_name()

return_grid = pd.concat(
    [
        bridge[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
        regular[["ticker", "trading_date", "bar_start_utc", "window_complete", "ohlc_valid", "nonpositive_price", "split_event", "return_requires_review", "log_ret_oc"]],
    ],
    ignore_index=True,
)
usable_return = (
    return_grid["window_complete"]
    & return_grid["ohlc_valid"]
    & ~return_grid["nonpositive_price"]
    & ~return_grid["split_event"]
    & ~return_grid["return_requires_review"]
)
return_grid.loc[~usable_return, "log_ret_oc"] = np.nan
return_grid["trading_date"] = return_grid["trading_date"].astype(str)
if return_grid.duplicated(["ticker", "bar_start_utc"]).any():
    raise ValueError("Combined bridge/regular return grid has duplicate keys")

return_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["log_ret_oc"]
date_lookup = return_grid.set_index(["ticker", "bar_start_utc"])["trading_date"]
risk_value_lookup = risk.set_index("hour")["AI_Risk_total"]
risk_eligible_lookup = risk.set_index("hour")["observed_eligible"]

for lag in range(1, 4):
    return_key_time = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    return_keys = pd.MultiIndex.from_arrays([targets["ticker"], return_key_time])
    targets[f"return_lag_{lag}"] = return_lookup.reindex(return_keys).to_numpy()
    lag_dates = date_lookup.reindex(return_keys).to_numpy()
    same_day = lag_dates == targets["trading_date"].to_numpy()
    targets.loc[~same_day, f"return_lag_{lag}"] = np.nan

    risk_hour = targets["bar_start_utc"] - pd.Timedelta(hours=lag)
    targets[f"risk_hour_lag_{lag}"] = risk_hour
    targets[f"risk_lag_{lag}"] = risk_value_lookup.reindex(risk_hour).to_numpy()
    eligible = risk_eligible_lookup.reindex(risk_hour)
    targets[f"risk_eligible_lag_{lag}"] = eligible.fillna(False).astype(bool).to_numpy()

targets = targets.sort_values(["bar_start_utc", "ticker"]).reset_index(drop=True)

assert (targets["risk_hour_lag_1"] + pd.Timedelta(hours=1) == targets["bar_start_utc"]).all()
assert not targets.duplicated(["ticker", "bar_start_utc"]).any()

audit = pd.DataFrame(
    [
        ("regular target rows", len(targets)),
        ("tickers", targets["ticker"].nunique()),
        ("regular trading dates", targets["trading_date"].nunique()),
        ("bridge expected rows", len(bridge)),
        ("bridge complete rows", int(bridge["window_complete"].sum())),
        ("risk hourly rows", len(risk)),
        ("risk observed-sample eligible rows", int(risk["observed_eligible"].sum())),
        ("risk strict-coverage eligible rows", int(risk["strict_eligible"].sum())),
        ("risk queries at result cap", int(pd.to_numeric(risk.get("queries_at_result_cap"), errors="coerce").max())),
    ],
    columns=["check", "value"],
)
display(audit)

example_row = targets.loc[
    targets["ticker"].eq("MSFT")
    & targets["target_start_ny"].dt.hour.eq(10)
    & targets["risk_lag_3"].notna()
].iloc[0]
alignment_example = pd.DataFrame(
    {
        "role": ["target return", "lag 1 risk/return", "lag 2 risk/return", "lag 3 risk/return"],
        "window_start_NY": [
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
            example_row["target_start_ny"] - pd.Timedelta(hours=3),
        ],
        "window_end_NY": [
            example_row["target_end_ny"],
            example_row["target_start_ny"],
            example_row["target_start_ny"] - pd.Timedelta(hours=1),
            example_row["target_start_ny"] - pd.Timedelta(hours=2),
        ],
    }
)
display(Markdown("**Exact no-look-ahead example (MSFT):**"))
display(alignment_example)

,check,value
0,regular target rows,2760
1,tickers,20
2,regular trading dates,23
3,bridge expected rows,1380
4,bridge complete rows,887
5,risk hourly rows,723
6,risk observed-sample eligible rows,622
7,risk strict-coverage eligible rows,0
8,risk queries at result cap,33


**Exact no-look-ahead example (MSFT):**

,role,window_start_NY,window_end_NY
0,target return,2026-09-02 10:00:00-04:00,2026-09-02 11:00:00-04:00
1,lag 1 risk/return,2026-09-02 09:00:00-04:00,2026-09-02 10:00:00-04:00
2,lag 2 risk/return,2026-09-02 08:00:00-04:00,2026-09-02 09:00:00-04:00
3,lag 3 risk/return,2026-09-02 07:00:00-04:00,2026-09-02 08:00:00-04:00


## 2. Individual-stock specification

Each stock is estimated separately; no return averaging or shared stock coefficient is used.

For every ticker:

1. Select its own return lag order $q\in\{1,2,3\}$ by BIC on a common return-only sample.
2. Hold that $q$ fixed and estimate risk lag orders $p=1,2,3$.
3. Use one common $p=3$-eligible sample for all three values of $p$ for that ticker.
4. Include New York target-hour and weekday fixed effects.
5. Use trading-date clustered inference as the primary test, with Newey–West HAC (`maxlags=5`) as a sensitivity check.
6. Correct the 20×3 = 60 primary p-values together using Benjamini–Hochberg FDR.

The same strict timestamp rule remains in force: risk from $[t-1,t)$ or earlier predicts return in $[t,t+1)$; neither missing risk nor missing bridge returns are filled.

## 3. Select a return lag order separately for each stock

For a given stock, its three BIC candidates use exactly the same observations: rows with valid within-day return lags through three hours.

In [3]:
REQUESTED_TICKERS = (
    "MSFT", "GOOGL", "META", "AMZN", "TSLA", "ORCL", "CRM", "NOW", "ADBE", "WDAY",
    "INTU", "PATH", "AAPL", "PLTR", "UBER", "SNOW", "SHOP", "RBLX", "SNAP", "PINS",
)
observed_tickers = set(targets["ticker"].unique())
if observed_tickers != set(REQUESTED_TICKERS):
    missing = sorted(set(REQUESTED_TICKERS) - observed_tickers)
    extra = sorted(observed_tickers - set(REQUESTED_TICKERS))
    raise ValueError(f"Ticker pool mismatch; missing={missing}, extra={extra}")
if not regular["duration_minutes"].eq(60).all():
    raise ValueError("Regular-return input contains a non-60-minute window")
if not bridge["duration_minutes"].eq(60).all():
    raise ValueError("Bridge-return input contains a non-60-minute window")


def individual_fixed_effects(frame: pd.DataFrame) -> pd.DataFrame:
    categories = frame[["target_hour", "weekday"]].astype(str)
    return pd.get_dummies(
        categories,
        columns=["target_hour", "weekday"],
        prefix=["hour", "weekday"],
        drop_first=True,
        dtype=float,
    )


def individual_design(frame: pd.DataFrame, q: int, p: int = 0) -> pd.DataFrame:
    matrix = pd.DataFrame({"const": np.ones(len(frame), dtype=float)}, index=frame.index)
    for lag in range(1, q + 1):
        matrix[f"return_lag_{lag}"] = frame[f"return_lag_{lag}"].astype(float)
    for lag in range(1, p + 1):
        matrix[f"risk_lag_{lag}"] = frame[f"risk_lag_{lag}"].astype(float)
    return pd.concat([matrix, individual_fixed_effects(frame)], axis=1).astype(float)


stock_bic_rows = []
for ticker, stock_frame in targets.groupby("ticker", sort=True):
    common_mask = stock_frame["target_return"].notna()
    for lag in range(1, 4):
        common_mask &= stock_frame[f"return_lag_{lag}"].notna()
    common = stock_frame.loc[common_mask].sort_values("bar_start_utc").copy()
    if common.empty:
        raise ValueError(f"No common q=1..3 BIC sample for {ticker}")
    for q in (1, 2, 3):
        X = individual_design(common, q=q, p=0)
        if np.linalg.matrix_rank(X.to_numpy()) != X.shape[1]:
            raise ValueError(f"Rank-deficient BIC design for {ticker}, q={q}")
        fit = sm.OLS(common["target_return"].astype(float), X, missing="raise").fit()
        stock_bic_rows.append(
            {
                "ticker": ticker,
                "q_return_lags": q,
                "n_obs_common_sample": int(fit.nobs),
                "n_parameters": int(fit.df_model + 1),
                "BIC": float(fit.bic),
                "adjusted_R2": float(fit.rsquared_adj),
            }
        )

stock_bic_results = pd.DataFrame(stock_bic_rows)
stock_bic_results["delta_BIC_within_ticker"] = (
    stock_bic_results["BIC"] - stock_bic_results.groupby("ticker")["BIC"].transform("min")
)
stock_q_choices = (
    stock_bic_results.sort_values(["ticker", "BIC", "q_return_lags"])
    .groupby("ticker", as_index=False)
    .first()[["ticker", "q_return_lags", "n_obs_common_sample", "BIC"]]
    .rename(columns={"q_return_lags": "selected_q", "BIC": "selected_BIC"})
)

display(Markdown("### 每只股票的BIC选择"))
display(
    stock_q_choices.style
    .format({"selected_BIC": "{:.3f}"})
    .hide(axis="index")
)
display(Markdown("### 收益滞后阶数分布"))
display(
    stock_q_choices["selected_q"]
    .value_counts()
    .sort_index()
    .rename_axis("selected_q")
    .rename("股票数量")
    .to_frame()
)

### 每只股票的BIC选择

ticker,selected_q,n_obs_common_sample,selected_BIC
AAPL,1,138,-1072.486
ADBE,1,135,-956.736
AMZN,1,138,-1109.041
CRM,1,69,-516.550
GOOGL,1,138,-1060.502
INTU,3,115,-831.041
META,1,138,-960.712
MSFT,1,138,-1105.082
NOW,3,69,-503.021
ORCL,1,69,-492.607


### 收益滞后阶数分布

,股票数量
selected_q,
1,16
2,1
3,3


## 4. Run 60 individual-stock Granger tests

For each stock and each $p$, the null is $H_0:\beta_1=\cdots=\beta_p=0$. The primary p-value comes from a small-cluster robust F test clustered by trading date. The HAC p-value is shown only as a sensitivity check. Effects are converted from log returns to percent returns for readability.

For $p>1$, the reported +1 SD cumulative effect means that all included lagged risk values increase by one standard deviation together. It is not a one-hour impulse response.

In [4]:
RISK_SD = float(risk.loc[risk["observed_eligible"], "AI_Risk_total"].std(ddof=1))
if not np.isfinite(RISK_SD) or RISK_SD <= 0:
    raise ValueError("AI_Risk_total has no usable variation")


def individual_common_sample(frame: pd.DataFrame, ticker: str, q: int, max_p: int = 3) -> pd.DataFrame:
    stock = frame.loc[frame["ticker"].eq(ticker)].copy()
    mask = stock["target_return"].notna()
    for lag in range(1, q + 1):
        mask &= stock[f"return_lag_{lag}"].notna()
    for lag in range(1, max_p + 1):
        mask &= stock[f"risk_lag_{lag}"].notna() & stock[f"risk_eligible_lag_{lag}"]
    return stock.loc[mask].sort_values("bar_start_utc").reset_index(drop=True)


def fit_individual_model(sample: pd.DataFrame, ticker: str, q: int, p: int) -> dict:
    y = sample["target_return"].astype(float)
    X_restricted = individual_design(sample, q=q, p=0)
    X_unrestricted = individual_design(sample, q=q, p=p)
    if len(sample) <= X_unrestricted.shape[1] + 20:
        raise ValueError(f"Insufficient residual degrees of freedom for {ticker}, q={q}, p={p}")
    if np.linalg.matrix_rank(X_unrestricted.to_numpy()) != X_unrestricted.shape[1]:
        raise ValueError(f"Rank-deficient individual design for {ticker}, q={q}, p={p}")

    restricted = sm.OLS(y, X_restricted, missing="raise").fit()
    unrestricted = sm.OLS(y, X_unrestricted, missing="raise").fit()
    names = list(X_unrestricted.columns)
    risk_positions = [names.index(f"risk_lag_{lag}") for lag in range(1, p + 1)]
    restriction = np.zeros((p, len(names)), dtype=float)
    for row_number, position in enumerate(risk_positions):
        restriction[row_number, position] = 1.0

    trading_day_groups = pd.Categorical(sample["trading_date"]).codes
    day_clustered = unrestricted.get_robustcov_results(
        cov_type="cluster",
        groups=trading_day_groups,
        use_correction=True,
        df_correction=True,
    )
    day_wald = day_clustered.wald_test(restriction, use_f=True, scalar=True)

    hac = unrestricted.get_robustcov_results(
        cov_type="HAC",
        maxlags=5,
        use_correction=True,
    )
    hac_wald = hac.wald_test(restriction, use_f=True, scalar=True)

    params = np.asarray(day_clustered.params, dtype=float)
    covariance = np.asarray(day_clustered.cov_params(), dtype=float)
    sum_vector = np.zeros(len(names), dtype=float)
    sum_vector[risk_positions] = 1.0
    beta_sum = float(sum_vector @ params)
    beta_sum_variance = float(sum_vector @ covariance @ sum_vector)
    beta_sum_se = np.sqrt(beta_sum_variance) if beta_sum_variance >= 0 else np.nan
    inference_df = float(day_wald.df_denom)
    critical_value = float(stats.t.ppf(0.975, inference_df))
    ci_low = beta_sum - critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan
    ci_high = beta_sum + critical_value * beta_sum_se if np.isfinite(beta_sum_se) else np.nan

    return {
        "ticker": ticker,
        "return_lags_q": q,
        "risk_lags_p": p,
        "n_obs": int(unrestricted.nobs),
        "n_target_timestamps": int(sample["bar_start_utc"].nunique()),
        "n_trading_days": int(sample["trading_date"].nunique()),
        "n_parameters": int(unrestricted.df_model + 1),
        "residual_df": int(unrestricted.df_resid),
        "risk_beta_sum": beta_sum,
        "effect_1SD_percent": 100 * np.expm1(beta_sum * RISK_SD),
        "effect_1SD_CI_low_percent": 100 * np.expm1(ci_low * RISK_SD),
        "effect_1SD_CI_high_percent": 100 * np.expm1(ci_high * RISK_SD),
        "day_cluster_F": float(np.asarray(day_wald.statistic).squeeze()),
        "day_cluster_df_num": p,
        "day_cluster_df_denom": inference_df,
        "day_cluster_p": float(np.asarray(day_wald.pvalue).squeeze()),
        "HAC5_F": float(np.asarray(hac_wald.statistic).squeeze()),
        "HAC5_p": float(np.asarray(hac_wald.pvalue).squeeze()),
        "adjusted_R2": float(unrestricted.rsquared_adj),
    }


individual_rows = []
individual_samples = {}
for choice in stock_q_choices.itertuples(index=False):
    ticker = str(choice.ticker)
    q = int(choice.selected_q)
    common = individual_common_sample(targets, ticker=ticker, q=q, max_p=3)
    individual_samples[ticker] = common
    for p in (1, 2, 3):
        individual_rows.append(fit_individual_model(common, ticker=ticker, q=q, p=p))

individual_results = pd.DataFrame(individual_rows).sort_values(["ticker", "risk_lags_p"]).reset_index(drop=True)
if len(individual_results) != 60 or individual_results["ticker"].nunique() != 20:
    raise ValueError("Expected exactly 20 tickers x 3 risk-lag specifications = 60 tests")
individual_results["BH_q_all_60"] = multipletests(individual_results["day_cluster_p"], method="fdr_bh")[1]
individual_results["Holm_p_all_60"] = multipletests(individual_results["day_cluster_p"], method="holm")[1]
individual_results["raw_p_below_0_05"] = individual_results["day_cluster_p"] < 0.05
individual_results["FDR_5pct_significant"] = individual_results["BH_q_all_60"] < 0.05
individual_results["Holm_5pct_significant"] = individual_results["Holm_p_all_60"] < 0.05

for p in (1, 2, 3):
    mask = individual_results["risk_lags_p"].eq(p)
    individual_results.loc[mask, "BH_q_within_p"] = multipletests(
        individual_results.loc[mask, "day_cluster_p"], method="fdr_bh"
    )[1]

individual_display = individual_results[[
    "ticker", "return_lags_q", "risk_lags_p", "n_obs", "n_trading_days",
    "risk_beta_sum", "effect_1SD_percent", "effect_1SD_CI_low_percent",
    "effect_1SD_CI_high_percent", "day_cluster_F", "day_cluster_df_denom",
    "day_cluster_p", "BH_q_all_60", "Holm_p_all_60", "FDR_5pct_significant", "HAC5_p",
]].rename(
    columns={
        "ticker": "股票",
        "return_lags_q": "收益滞后q",
        "risk_lags_p": "风险滞后p",
        "n_obs": "观测数",
        "n_trading_days": "交易日数",
        "risk_beta_sum": "风险系数合计",
        "effect_1SD_percent": "风险+1SD收益影响(%)",
        "effect_1SD_CI_low_percent": "95%CI下限(%)",
        "effect_1SD_CI_high_percent": "95%CI上限(%)",
        "day_cluster_F": "按交易日聚类F",
        "day_cluster_df_denom": "分母df",
        "day_cluster_p": "原始p值",
        "BH_q_all_60": "60项BH q值",
        "Holm_p_all_60": "60项Holm校正p值",
        "FDR_5pct_significant": "FDR 5%显著",
        "HAC5_p": "HAC(5) p值",
    }
)

display(Markdown("### 全部60项个股检验"))
display(
    individual_display.style.format(
        {
            "风险系数合计": "{:.8f}",
            "风险+1SD收益影响(%)": "{:.5f}",
            "95%CI下限(%)": "{:.5f}",
            "95%CI上限(%)": "{:.5f}",
            "按交易日聚类F": "{:.4f}",
            "分母df": "{:.0f}",
            "原始p值": "{:.6g}",
            "60项BH q值": "{:.6g}",
            "60项Holm校正p值": "{:.6g}",
            "HAC(5) p值": "{:.6g}",
        }
    ).hide(axis="index")
)

### 全部60项个股检验

股票,收益滞后q,风险滞后p,观测数,交易日数,风险系数合计,风险+1SD收益影响(%),95%CI下限(%),95%CI上限(%),按交易日聚类F,分母df,原始p值,60项BH q值,60项Holm校正p值,FDR 5%显著,HAC(5) p值
AAPL,1,1,95,20,0.00014610,0.02070,-0.06518,0.10666,0.2543,19,0.61983,0.919771,1,False,0.611039
AAPL,1,2,95,20,0.00052689,0.07468,-0.03297,0.18244,1.8445,19,0.185318,0.694944,1,False,0.228942
AAPL,1,3,95,20,0.00060486,0.08573,-0.04864,0.22028,1.3254,19,0.29552,0.806708,1,False,0.41113
ADBE,1,1,95,20,0.00033955,0.04812,-0.04509,0.14142,1.1669,19,0.293558,0.806708,1,False,0.328562
ADBE,1,2,95,20,0.00008691,0.01231,-0.13553,0.16037,1.4861,19,0.251406,0.793913,1,False,0.334769
ADBE,1,3,95,20,-0.00018098,-0.02564,-0.23437,0.18353,1.6116,19,0.219888,0.766184,1,False,0.179092
AMZN,1,1,95,20,-0.00005278,-0.00748,-0.07628,0.06138,0.0517,19,0.822579,0.922879,1,False,0.813392
AMZN,1,2,95,20,0.00007132,0.01010,-0.08064,0.10093,0.3907,19,0.68191,0.922879,1,False,0.702184
AMZN,1,3,95,20,0.00012299,0.01743,-0.08731,0.12228,0.3633,19,0.780228,0.922879,1,False,0.850088
CRM,1,1,84,20,0.00070016,0.09925,-0.01399,0.21261,3.3649,19,0.0823082,0.511755,1,False,0.0632877


## 5. Multiple-testing summary

Raw p-values are shown because they reveal which stocks are worth examining, but the pre-specified overall conclusion uses the BH-adjusted q-values across all 60 tests.

In [5]:
raw_p_matrix = individual_results.pivot(index="ticker", columns="risk_lags_p", values="day_cluster_p")
raw_p_matrix.columns = [f"p={int(value)}" for value in raw_p_matrix.columns]
fdr_matrix = individual_results.pivot(index="ticker", columns="risk_lags_p", values="BH_q_all_60")
fdr_matrix.columns = [f"p={int(value)}" for value in fdr_matrix.columns]

display(Markdown("### 原始p值矩阵"))
display(raw_p_matrix.style.format("{:.4f}"))
display(Markdown("### 对全部60项统一BH校正后的q值矩阵"))
display(fdr_matrix.style.format("{:.4f}"))

raw_significant = individual_results.loc[individual_results["raw_p_below_0_05"]].sort_values("day_cluster_p")
fdr_significant = individual_results.loc[individual_results["FDR_5pct_significant"]].sort_values("BH_q_all_60")

display(Markdown("### 原始p < 0.05的结果（未经多重检验校正）"))
if raw_significant.empty:
    display(Markdown("没有。"))
else:
    display(
        raw_significant[[
            "ticker", "return_lags_q", "risk_lags_p", "n_obs", "effect_1SD_percent",
            "day_cluster_F", "day_cluster_p", "BH_q_all_60", "Holm_p_all_60", "HAC5_p",
        ]].style.format(
            {
                "effect_1SD_percent": "{:.5f}",
                "day_cluster_F": "{:.4f}",
                "day_cluster_p": "{:.6g}",
                "BH_q_all_60": "{:.6g}",
                "Holm_p_all_60": "{:.6g}",
                "HAC5_p": "{:.6g}",
            }
        ).hide(axis="index")
    )

display(Markdown("### BH-FDR 5%校正后显著的结果"))
if fdr_significant.empty:
    display(Markdown("没有。"))
else:
    display(fdr_significant)

summary_table = pd.DataFrame(
    {
        "指标": [
            "完成的个股检验", "原始p<0.05", "全60项BH后q<0.05",
            "最小原始p值", "最小BH q值", "最小Holm校正p值", "个股共同样本最小N", "个股共同样本最大N",
        ],
        "数值": [
            len(individual_results),
            int(individual_results["raw_p_below_0_05"].sum()),
            int(individual_results["FDR_5pct_significant"].sum()),
            float(individual_results["day_cluster_p"].min()),
            float(individual_results["BH_q_all_60"].min()),
            float(individual_results["Holm_p_all_60"].min()),
            int(individual_results["n_obs"].min()),
            int(individual_results["n_obs"].max()),
        ],
    }
)
display(Markdown("### 个股检验摘要"))
display(summary_table)

if fdr_significant.empty:
    conclusion = (
        f"共完成{len(individual_results)}项个股检验；"
        f"{int(individual_results['raw_p_below_0_05'].sum())}项原始p值低于0.05，"
        "但对全部60项进行BH校正后没有结果达到5% FDR。"
    )
else:
    labels = ", ".join(
        f"{row.ticker}(p={int(row.risk_lags_p)})" for row in fdr_significant.itertuples()
    )
    conclusion = f"BH校正后达到5% FDR的个股模型为：{labels}。"

display(Markdown("### 个股检验的机械性结论"))
display(Markdown(conclusion))
display(
    Markdown(
        "这些仍然是 observed X News sample 中的预测关联。个股样本较短，"
        "原始显著而FDR不显著的结果不能作为确认性发现。"
    )
)

### 原始p值矩阵

,p=1,p=2,p=3
ticker,,,
AAPL,0.6198,0.1853,0.2955
ADBE,0.2936,0.2514,0.2199
AMZN,0.8226,0.6819,0.7802
CRM,0.0823,0.1555,0.1461
GOOGL,0.6285,0.8306,0.9328
INTU,0.0853,0.0545,0.1187
META,0.4930,0.7542,0.7635
MSFT,0.0235,0.0514,0.0852
NOW,0.0058,0.0103,0.0183


### 对全部60项统一BH校正后的q值矩阵

,p=1,p=2,p=3
ticker,,,
AAPL,0.9198,0.6949,0.8067
ADBE,0.8067,0.7939,0.7662
AMZN,0.9229,0.9229,0.9229
CRM,0.5118,0.6316,0.6316
GOOGL,0.9198,0.9229,0.9936
INTU,0.5118,0.5118,0.6316
META,0.9198,0.9229,0.9229
MSFT,0.3528,0.5118,0.5118
NOW,0.3100,0.3100,0.3528


### 原始p < 0.05的结果（未经多重检验校正）

...

### 个股检验摘要

,指标,数值
0,完成的个股检验,60
1,原始p<0.05,4
2,全60项BH后q<0.05,0
3,最小原始p值,0.00577958
4,最小BH q值,0.310048
5,最小Holm校正p值,0.346775
6,个股共同样本最小N,54
7,个股共同样本最大N,95


### 个股检验的机械性结论

共完成60项个股检验；4项原始p值低于0.05，但对全部60项进行BH校正后没有结果达到5% FDR。

这些仍然是 observed X News sample 中的预测关联。个股样本较短，原始显著而FDR不显著的结果不能作为确认性发现。